# N12 — Overtake Probability Model

**LightGBM binary classifier** — `P(overtake | lap features)`

Trained on the labeled pair dataset produced by N11. Predicts whether car X overtakes car Y **on the current lap**, given the observable state of the pair at lap boundary.

| | |
|---|---|
| Input | `data/processed/overtake_labeled/overtake_pairs_2023_2025.parquet` |
| Train | 2023 + 2024 |
| Test | 2025 |
| Export | `data/models/overtake_probability/` |

---

## Step 0 — Setup

Standard imports plus LightGBM and SHAP. Paths follow the same convention as N11: `repo_root` resolved via `.git` walker, outputs split between the notebook's `outputs/` folder and `data/models/` for exportable artifacts.


In [1]:
# ── Step 0 · Setup ────────────────────────────────────────────────────────────
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report,
)
from sklearn.calibration import calibration_curve
import shap


In [2]:
# ── repo root ─────────────────────────────────────────────────────────────────
repo_root = Path().resolve()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks/strategy/overtake_probability/outputs"
PROCESSED  = repo_root / "data/processed/overtake_labeled"
EXPORT_DIR = repo_root / "data/models/overtake_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("repo_root  :", repo_root)
print("OUTPUTS    :", OUTPUTS)
print("PROCESSED  :", PROCESSED)
print("EXPORT_DIR :", EXPORT_DIR)
print("\nlightgbm   :", lgb.__version__)
print("shap       :", shap.__version__)

repo_root  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
OUTPUTS    : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\overtake_probability\outputs
PROCESSED  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\overtake_labeled
EXPORT_DIR : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\overtake_probability

lightgbm   : 4.6.0
shap       : 0.49.1


---